In [1]:
import numpy as np
import pandas as pd
import torch
from torch import nn
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import normalize
from sklearn.metrics import confusion_matrix

In [2]:
class SingleLayer(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        self.sequential = nn.Sequential(
            nn.Linear(input_size, 8),
            nn.ReLU(),
            #nn.Dropout(0.25), # works slightly better without the dropout layer, and considering we're already doing mini-batching and a train-test split, I'm not too concerned about overfitting
            nn.Linear(8, 2))
    def forward(self, x):
        return(self.sequential(x))

In [3]:
mystery_data = pd.read_csv("../../data/FP_Data.csv")

mystery_data_onehot = pd.get_dummies(mystery_data)

y = mystery_data_onehot.pop("y")
y = np.array([1 if obs >= 70 else 0 for obs in y])
X = mystery_data_onehot

X = normalize(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.75, random_state=28)

In [4]:
model = SingleLayer(11)
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1) # This is, as far as I seen, the most widely used optimizer, though it is not what is used in the textbook
loss_fn = nn.CrossEntropyLoss()
#log_softmax = nn.functional.log_softmax(dim = 1, dtype = torch.float32)

epochs = 50
batch_size = 32

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(np.asarray(y_train), dtype=torch.long)
y_test = torch.tensor(np.asarray(y_test), dtype=torch.long)

for epoch in range(epochs):
    model.train()

    permutation = torch.randperm(X_train.size(0))
    for i in range(0, X_train.size(0), batch_size):
        indices = permutation[i:i+batch_size]
        X_batch, y_batch = X_train[indices], y_train[indices]

        optimizer.zero_grad()
        output = model(X_batch)
        loss = loss_fn(output, y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_output = model(X_test)
        val_loss = loss_fn(val_output, y_test)
    print(f"Epoch {epoch+1}/{epochs} | Val Loss: {val_loss.item():.4f}")

Epoch 1/50 | Val Loss: 0.3142
Epoch 2/50 | Val Loss: 0.2880
Epoch 3/50 | Val Loss: 0.2879
Epoch 4/50 | Val Loss: 0.3080
Epoch 5/50 | Val Loss: 0.3131
Epoch 6/50 | Val Loss: 0.3176
Epoch 7/50 | Val Loss: 0.3276
Epoch 8/50 | Val Loss: 0.3203
Epoch 9/50 | Val Loss: 0.3220
Epoch 10/50 | Val Loss: 0.3278
Epoch 11/50 | Val Loss: 0.3493
Epoch 12/50 | Val Loss: 0.3834
Epoch 13/50 | Val Loss: 0.4132
Epoch 14/50 | Val Loss: 0.4496
Epoch 15/50 | Val Loss: 0.4634
Epoch 16/50 | Val Loss: 0.4590
Epoch 17/50 | Val Loss: 0.4662
Epoch 18/50 | Val Loss: 0.4963
Epoch 19/50 | Val Loss: 0.5105
Epoch 20/50 | Val Loss: 0.5516
Epoch 21/50 | Val Loss: 0.5534
Epoch 22/50 | Val Loss: 0.5836
Epoch 23/50 | Val Loss: 0.5916
Epoch 24/50 | Val Loss: 0.5861
Epoch 25/50 | Val Loss: 0.6149
Epoch 26/50 | Val Loss: 0.6045
Epoch 27/50 | Val Loss: 0.6122
Epoch 28/50 | Val Loss: 0.6074
Epoch 29/50 | Val Loss: 0.5961
Epoch 30/50 | Val Loss: 0.6217
Epoch 31/50 | Val Loss: 0.5944
Epoch 32/50 | Val Loss: 0.5930
Epoch 33/50 | Val

In [5]:
model.eval()
y_test = y_test.detach().numpy()
with torch.no_grad():
    test_pred = model(X_test).argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()
    confusion = confusion_matrix(y_test, test_pred)
    print(f"Test Accuracy: {test_acc:.4f}")

Test Accuracy: 0.8000


## 10-Fold Cross Validation

In [6]:
df = pd.read_csv("../04-cv/10-fold-cv.csv")

torch.manual_seed(28)
np.random.seed(28)

df_onehot = pd.get_dummies(df.drop(columns=["fold"]))
y = df_onehot.pop("y")
y = np.array([1 if obs >= 70 else 0 for obs in y])
X = normalize(df_onehot)

k = 10
epochs = 50
batch_size = 32
nn_cv_results = []

In [7]:
for fold in range(1, k + 1):

    train_mask = (df["fold"] != fold).values
    test_mask = (df["fold"] == fold).values

    X_train = torch.tensor(X[train_mask], dtype=torch.float32)
    X_test = torch.tensor(X[test_mask], dtype=torch.float32)
    y_train = torch.tensor(np.asarray(y[train_mask]), dtype=torch.long)
    y_test = torch.tensor(np.asarray(y[test_mask]), dtype=torch.long)

    model = SingleLayer(11)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()

        permutation = torch.randperm(X_train.size(0))
        for i in range(0, X_train.size(0), batch_size):
            indices = permutation[i:i+batch_size]
            X_batch, y_batch = X_train[indices], y_train[indices]

            optimizer.zero_grad()
            output = model(X_batch)
            loss = loss_fn(output, y_batch)
            loss.backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            val_output = model(X_test)
            val_loss = loss_fn(val_output, y_test)

    model.eval()
    y_test_np = y_test.detach().numpy()
    with torch.no_grad():
        test_pred = model(X_test).argmax(dim=1)
        accuracy = (test_pred == y_test).float().mean().item()
        confusion = confusion_matrix(y_test_np, test_pred)

    tn, fp, fn, tp = confusion.ravel()
    fpr = fp / (fp + tn)
    fnr = fn / (fn + tp)

    nn_cv_results.append({"fold": fold, "model": "NeuralNetwork", "accuracy": accuracy, "fpr": fpr, "fnr": fnr})

nn_cv_df = pd.DataFrame(nn_cv_results)
nn_cv_df.to_csv("../04-cv/class_nn_cv_results.csv", index=False)
print(nn_cv_df)

   fold          model  accuracy       fpr   fnr
0     1  NeuralNetwork  0.809524  0.210526  0.00
1     2  NeuralNetwork  0.809524  0.055556  1.00
2     3  NeuralNetwork  0.809524  0.066667  0.50
3     4  NeuralNetwork  0.761905  0.117647  0.75
4     5  NeuralNetwork  0.809524  0.176471  0.25
5     6  NeuralNetwork  0.600000  0.266667  0.80
6     7  NeuralNetwork  0.850000  0.125000  0.25
7     8  NeuralNetwork  0.736842  0.133333  0.75
8     9  NeuralNetwork  0.833333  0.142857  0.25
9    10  NeuralNetwork  0.944444  0.058824  0.00
